# E5 — learning rate and augmentation ablation

Owner: Nguyễn Ngọc Kỳ. Select **T4 GPU**, then **Run all**. Training is seed 42 only.
Public clone; no token required. Allow Drive mount to preserve checkpoints.
No test-set evaluation is used for model or hyperparameter selection.
Historical outputs below were validated on T4 before PR31 was merged. The bootstrap cell was subsequently updated and its old output cleared: it now clones the immutable merged commit because the leader deleted the remote work branch. The new bootstrap was verified with a fresh public clone on Windows; the training code is unchanged.


In [ ]:
import os, subprocess, sys
from pathlib import Path
REPO = Path('/content/vit-oxford-pets')
SOURCE_COMMIT = '0dbb87734416501711695ccf3f8e0af882bd84f1'
WORK_BRANCH = 'ky/e3-e5-demo-report'
URL = 'https://github.com/duyenphung206-sudo/vit-oxford-pets.git'
if not REPO.exists():
    subprocess.run(['git', 'clone', '--no-checkout', URL, str(REPO)], check=True)
    subprocess.run(['git', '-C', str(REPO), 'switch', '-c', WORK_BRANCH, SOURCE_COMMIT], check=True)
os.chdir(REPO)
branch = subprocess.check_output(['git', 'branch', '--show-current'], text=True).strip()
if branch != WORK_BRANCH or not Path('src/ky_experiments.py').exists():
    raise RuntimeError('Preserve this existing checkout; use a fresh runtime with the Ky work branch.')
print('Source commit:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
os.environ['MPLCONFIGDIR'] = '/content/ky-cache/matplotlib'
os.environ['TORCH_HOME'] = '/content/ky-cache/torch'


In [2]:
import torch, torchvision
print('Torch:', torch.__version__, 'torchvision:', torchvision.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    if 'T4' not in torch.cuda.get_device_name(0):
        print('GPU differs from preferred T4; actual hardware will be recorded in metrics.')
else:
    print('BLOCKED: real E3/E5 runs await GPU. Only the synthetic pipeline check is permitted on CPU.')


Torch: 2.14.1+cu130 torchvision: 0.29.1+cu130
CUDA available: True
GPU: Tesla T4


In [3]:
# Synthetic outputs are isolated and never reported as experiment results.
import yaml
from src.utils import load_config
smoke_cfg = load_config('configs/e3_vit_partial.yaml')
smoke_cfg['experiment'] = 'ky_smoke_e3'
smoke_cfg['model']['pretrained'] = False
smoke_cfg['data'].update(batch_size=1, num_workers=0)
smoke_path = Path('/content/ky-smoke.yaml')
smoke_path.write_text(yaml.safe_dump(smoke_cfg))
subprocess.run([sys.executable, '-m', 'src.train', '--config', str(smoke_path),
                '--fake-data', '--epochs', '1', '--output-dir', '/content/ky-smoke/results',
                '--checkpoint-dir', '/content/ky-smoke/checkpoints'], check=True)


In [4]:
from src.models.factory import build_model
from src.utils import count_parameters, load_config
cfg = load_config('configs/e3_vit_partial.yaml')
cfg['model']['pretrained'] = False  # Structural check; the real training config keeps ImageNet weights.
model = build_model(cfg['model'])
for name, parameter in model.named_parameters():
    expected = name.startswith(('heads.', 'encoder.ln.')) or any(
        name.startswith(f'encoder.layers.encoder_layer_{i}.') for i in range(8, 12))
    assert parameter.requires_grad == expected, name
print(count_parameters(model))
assert count_parameters(model)['trainable'] == 28_381_477
del model


{'total': 85827109, 'trainable': 28381477}


In [5]:
SELECTED = ['E5-LR1', 'E5-LR2', 'E5-LR3', 'E5-AUG']
# E5-LR4 is optional: add it explicitly. E5-LR2 reuses E4 seed42 only.


In [6]:
# Keep checkpoints across Colab disconnects; Drive prompts once for authorization.
PERSIST_TO_DRIVE = True
if not torch.cuda.is_available():
    raise RuntimeError('BLOCKED: select Runtime > Change runtime type > T4 GPU, then Run all.')
if PERSIST_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    STORE = Path('/content/drive/MyDrive/vit-oxford-pets-ky')
    STORE.mkdir(parents=True, exist_ok=True)
else:
    STORE = Path('/content/ky-artifacts')
    STORE.mkdir(parents=True, exist_ok=True)
# Symlink only Ky outputs. Never replace E4 or other members' files.
from src.ky_experiments import CONFIGS, load_config, run_dir
for name in SELECTED:
    if name == 'E5-LR2':
        continue
    cfg = load_config(Path('configs') / CONFIGS[name])
    for category in ('results', 'checkpoints'):
        local = Path(category) / cfg['experiment']
        target = STORE / category / cfg['experiment']
        target.mkdir(parents=True, exist_ok=True)
        if not local.exists():
            local.parent.mkdir(parents=True, exist_ok=True)
            local.symlink_to(target, target_is_directory=True)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [7]:
from src.ky_experiments import run
for name in SELECTED:
    run(name)  # skips verified runs, rejects incomplete evidence, never retrains E4


Reuse verified evidence: results/e5_lr1/seed42
Reuse verified evidence: results/e4_vit_full_ft/seed42
Reuse verified evidence: results/e5_lr3/seed42
Reuse verified evidence: results/e5_aug/seed42


In [8]:
from src.ky_experiments import summarize
summarize()


[
  {
    "id": "E5-LR1",
    "learning_rate": 0.001,
    "augmentation": true,
    "status": "DONE",
    "val_accuracy": 0.913043,
    "val_loss": 1.13896,
    "training_time_s": 502.5,
    "epochs": 10,
    "seed": 42,
    "source": "results/e5_lr1/seed42",
    "reason": "Repository evidence verified"
  },
  {
    "id": "E5-LR2",
    "learning_rate": 0.003,
    "augmentation": true,
    "status": "DONE",
    "val_accuracy": 0.955163,
    "val_loss": 0.311232,
    "training_time_s": 538.6,
    "epochs": 10,
    "seed": 42,
    "source": "results/e4_vit_full_ft/seed42",
    "reason": "Repository evidence verified"
  },
  {
    "id": "E5-LR3",
    "learning_rate": 0.01,
    "augmentation": true,
    "status": "DONE",
    "val_accuracy": 0.956522,
    "val_loss": 0.158747,
    "training_time_s": 498.8,
    "epochs": 10,
    "seed": 42,
    "source": "results/e5_lr3/seed42",
    "reason": "Repository evidence verified"
  },
  {
    "id": "E5-LR4",
    "learning_rate": 0.03,
    "augmentat

In [9]:
# Small archive: verified logs, configs, metrics, plots. Checkpoints remain on Drive.
import zipfile
from src.ky_experiments import CONFIGS, load_config, run_dir, verified_run
archive = Path('/content/ky-results.zip')
with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as bundle:
    for name in SELECTED:
        cfg = load_config(Path('configs') / CONFIGS[name])
        directory = run_dir(cfg)
        if (directory / 'metrics.json').exists():
            verified_run(directory, cfg)
            for path in directory.iterdir():
                if path.is_file():
                    bundle.write(path, str(path))
    summary_dir = Path('results/e5_ablation')
    for pattern in ('summary_seed42.*', '*_comparison.png', '*_comparison.svg'):
        for path in summary_dir.glob(pattern):
            bundle.write(path, str(path))
print('Verified artifacts:', archive)
print('Checkpoint storage:', STORE / 'checkpoints')
print('Save downloaded files locally under C:/Users/Ken/Downloads only.')
from google.colab import files
files.download(str(archive))


Verified artifacts: /content/ky-results.zip
Checkpoint storage: /content/drive/MyDrive/vit-oxford-pets-ky/checkpoints
Save downloaded files locally under C:/Users/Ken/Downloads only.
Download callback validated; archive preserved: /content/drive/MyDrive/vit-oxford-pets-ky/notebook-validation/1795201573892802163-08_train_e5_ablation.ipynb.results.zip


## After execution
Save the downloaded archive in Downloads. Preserve Drive checkpoints; do not commit them. Before updating the PR, validate imported log/config/metrics with `verified_run`, rerun `python -m src.ky_experiments --summarize`, and inspect the diff. A partial run is preserved and must be reviewed manually; it is never overwritten.
